In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.3 Quantization

Quantization stores weights in fewer bits, 8-bit or 4-bit instead of 32, cutting
memory by 4-8x. It is not free: low precision costs some accuracy. The rule is to
*measure* the degradation, never assume it.

In [ ]:
import copy
from pocketlm import train_tiny, pick_config, fake_quantize, estimate_loss

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)
g = torch.Generator().manual_seed(0)
before = estimate_loss(model, data, model.cfg.block_size, 16, iters=10, generator=g)
qmodel = copy.deepcopy(model)
with torch.no_grad():
    for p in qmodel.parameters():
        p.copy_(fake_quantize(p, bits=4))
g2 = torch.Generator().manual_seed(0)
after = estimate_loss(qmodel, data, model.cfg.block_size, 16, iters=10, generator=g2)
print(f"loss fp32: {before:.3f}  ->  4-bit: {after:.3f}  (delta {after - before:+.3f})")

Four-bit weights take a quarter of the memory; the loss moves by the printed delta.
On real models 8-bit is nearly free and 4-bit is a deliberate trade, always backed
by a measured number like this one.

In [ ]:
import math
assert math.isfinite(after)
assert after > 0